### Single Chromosome Simulation for Classic MiChroM

## Reproducible execution modes

This notebook runs the scientifically complete workflow by default. Set the environment variable `OPENMICHROM_TUTORIAL_MODE=fast` for the reduced CPU validation used by `python scripts/validate.py complete`. Fast mode exercises the same setup, force construction, integration, analysis, and output paths with fewer steps; it is a smoke test, not a production simulation. Set `OPENMICHROM_TUTORIAL_PLATFORM` to override the default `CPU` platform.


In [ ]:
import os
from pathlib import Path

import numpy as np

TUTORIAL_FAST = os.environ.get("OPENMICHROM_TUTORIAL_MODE", "full").lower() == "fast"
TUTORIAL_PLATFORM = os.environ.get("OPENMICHROM_TUTORIAL_PLATFORM", "CPU")
np.random.seed(2026)
print(f"OpenMiChroM tutorial mode: {'fast smoke test' if TUTORIAL_FAST else 'full scientific run'}")
print(f"OpenMM platform: {TUTORIAL_PLATFORM}")


This tutorial should take between 10 to 20 minutes of reading and performing simulations.

#### Chromatin Dynamics Simulations on Chromosome 10 of stomach GRCh38 (from ENCODE Project)

**Note**: This tutorial will be running in OpenMiChroM version 1.1.0 or greater. Please ensure you have the correct version installed before proceeding.

The first step is to import the **OpenMiChroM** module

In [1]:
from OpenMiChroM.ChromDynamics import MiChroM

Download the bed file that contains the sequence annotation for stomach via ENCODE website (https://www.encodeproject.org/)

In [ ]:
bed_path = Path('inputs/ENCFF963ZUJ.bed')
if not bed_path.exists():
    raise FileNotFoundError(f'Missing tutorial input: {bed_path}')
print(f'Using {bed_path}')


`MiChroM` class sets the initial parameters of the simulation:

- `timeStep=0.01`: set the simulation time step to perfom the integration<br>
- `temperature=1.0`: set the temperature of your simulation<br>

In [ ]:
sim = MiChroM(name='stomach_GRCh38', temperature=1.0, timeStep=0.01)

There are four hardware platform options to run the simulations: 
```python
platform="cuda"
platform="opencl"
platform="hip"
platform="cpu"
```

Choose accordingly.

In [ ]:
sim.setup(platform=TUTORIAL_PLATFORM)

Set the directory name in which the output of the simulation is saved:

In [ ]:
output_dir = Path('output_classic_michrom')
sim.saveFolder(str(output_dir))

The next step is to load the chromatin compartment sequence for chromosome 10 and generate an initial 3D structure to start the simulation. We can use the [createSpringSpiral](https://open-michrom.readthedocs.io/en/latest/OpenMiChroM.html#OpenMiChroM.ChromDynamics.MiChroM.createSpringSpiral) function to set the initial configuration of the polymer based in the sequence file.

We will use the bed file download above, and set the chromosome 10 to slice the file and get the sequence annotation.

In [ ]:
sim.buildClassicMichrom(ChromSeq='inputs/ENCFF963ZUJ.bed', chromosome='chr10')


As you can see on the output above, we build the system with MiChroM Potential add the homopolymer potentials and the Michrom Potentials.

The system reports some statitics as, number of beads, number of chains and the initial energy potential for each force applied.

Now we create the reporters to save the simulation infos. There are 3 types of reporters:

**statistics**: Attaches a reporter to collect simulation statistics such as step number, radius of gyration (RG), total energy, potential energy, kinetic energy, and temperature.

**trajectory**:  Attaches a reporter to save trajectory data (xyz per bead per chain) during the simulation. The file format to save the trajectory data. Options are 'cndb', 'swb','ndb', 'pdb', 'gro', 'xyz'. (Default: 'cndb')

**energy components**: Saves energy components per force group to a separate file named 'energyComponents.txt' in the simulation folder. Requires that statistics is True

set the number of steps interval we will save this information, here I choose 1000 steps

In [6]:
sim.createReporters(statistics=True, traj=True, trajFormat="cndb", energyComponents=True, interval=10**3)


The `sim.run()` function is used to start the simulation. The parameters for this function are:

- `nsteps`: The number of steps to run the simulation. In this case, it is set to \(10^5\).
- `report`: A boolean value indicating whether to report the simulation progress. Here, it is set to `True`.
- `interval`: The interval at which the simulation reports progress. In this case, it is set to \(10^4\) steps.



In [ ]:
production_steps = 20 if TUTORIAL_FAST else 10**6
sim.run(nsteps=production_steps, report=True, interval=max(1, min(10**4, production_steps)))

After the simulation ends, you will find the generated files in the configured output directory ("output" in this tutorial).

In [ ]:
print((output_dir / 'initialStats.txt').read_text())

In [ ]:
print(''.join((output_dir / 'statistics.txt').read_text().splitlines(True)[:10]))

In [ ]:
print(''.join((output_dir / 'energyComponents.txt').read_text().splitlines(True)[:10]))